In [ ]:
import os
import glob
import pandas as pd
from datetime import date, datetime
from dateutil.relativedelta import relativedelta
import gc

# OPTIMIZATION: Set pandas options for memory efficiency
pd.set_option('mode.chained_assignment', None)

# Get current directory
current_directory = os.getcwd()

# Find all CSV files
csv_files = glob.glob(os.path.join(current_directory, "*_MT_SkuDescListing_*.csv"))

# OPTIMIZATION: Define expected dtypes to avoid mixed type warnings and reduce memory
dtype_spec = {
    'Line': 'object',
    'Division': 'object', 
    'Department': 'object',
    'Short SKU': 'str',
    'Description': 'object',
    'Creation Date': 'object',
    'Activated Date': 'object',
    'Retail W GST': 'float32',
    'Returnable?': 'object'
}

# Read files one by one with explicit dtypes
skudesc = []
for i, file in enumerate(csv_files):
    print(f"Processing SKU file {i+1}/{len(csv_files)}: {os.path.basename(file)}")
    
    # Read with specified dtypes and skip rows
    df = pd.read_csv(file, skiprows=2, dtype=dtype_spec, low_memory=False)
    
    # Clean string columns immediately
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str).str.replace('=\"', '', regex=False).str_replace('\"', '', regex=False)
    
    # Convert Short SKU to 8 characters
    df['Short SKU'] = df['Short SKU'].str.zfill(8)
    
    # Drop Unnamed columns
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    
    # Keep only needed columns early to save memory
    columns_to_keep = ['Line', 'Division', 'Department', 'Short SKU', 'Description', 
                      'Creation Date', 'Activated Date', 'Retail W GST', 'Returnable?']
    df = df[[col for col in columns_to_keep if col in df.columns]]
    
    skudesc.append(df)
    del df
    gc.collect()

# Combine all DataFrames
print("Combining SKU description files...")
skudesc_combined = pd.concat(skudesc, ignore_index=True)
del skudesc
gc.collect()

# Convert date columns to YYYYMM format
for col in ['Creation Date', 'Activated Date']:
    if col in skudesc_combined.columns:
        skudesc_combined[col] = pd.to_datetime(skudesc_combined[col], errors='coerce').dt.strftime('%Y%m')

# OPTIMIZATION: Convert categorical columns to category dtype for memory efficiency
categorical_cols = ['Line', 'Division', 'Department', 'Returnable?']
for col in categorical_cols:
    if col in skudesc_combined.columns:
        skudesc_combined[col] = skudesc_combined[col].astype('category')

print(f"Loaded {len(skudesc_combined)} SKU records")

In [ ]:
# Columns already filtered for memory efficiency
print(f"Columns: {skudesc_combined.columns.tolist()}")
print(f"Memory usage: {skudesc_combined.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Use static period (as in original)
today_month = "202603"

# Add Period column
skudesc_combined['Period'] = today_month

# Display sample
print(skudesc_combined.head())

In [ ]:
import os
import pandas as pd
import glob
import tempfile
import shutil
import gc

# Configuration
current_dir = os.getcwd()
csv_files = glob.glob(os.path.join(current_dir, "*Daily Sales*.csv"))

# Create temp directory for intermediate processing
temp_dir = tempfile.mkdtemp()
print(f"Temp directory: {temp_dir}")

processed_files = []

# Stage 1: Process each file sequentially & save temporarily
for i, file in enumerate(csv_files):
    print(f"Processing file {i+1}/{len(csv_files)}: {os.path.basename(file)}")
    
    try:
        # OPTIMIZATION: Read with chunking for large files
        chunk_size = 100000
        chunks = []
        
        for chunk in pd.read_csv(file, skiprows=[0,1,2,3,4,5,6,8], chunksize=chunk_size):
            if 'Date' not in chunk.columns or 'Item' not in chunk.columns:
                continue
            
            chunk['Date'] = chunk['Date'].astype(str).str.replace(r'[\"=]', '', regex=True)
            chunk = chunk.rename(columns={'Item': 'Short SKU'})
            chunk['Short SKU'] = chunk['Short SKU'].astype(str).str.replace(r'[\"=]', '', regex=True)
            chunk['Short SKU'] = chunk['Short SKU'].str.rjust(8, '0')
            
            cols_to_keep = ['Date', 'Short SKU'] + [
                col for col in chunk.columns 
                if str(col).startswith(('10', '51', '70'))
            ]
            chunk = chunk[cols_to_keep]
            
            raisa = [(1001, 5101), (1002, 5102), (1003, 5103), (1004, 5104), (1006, 5106)]
            for offline, online in raisa:
                off_str, on_str = str(offline), str(online)
                if off_str in chunk.columns and on_str in chunk.columns:
                    chunk[off_str] = chunk[off_str].fillna(0) + chunk[on_str].fillna(0)
                    chunk = chunk.drop(columns=[on_str])
            
            chunks.append(chunk)
            del chunk
        
        if chunks:
            df_file = pd.concat(chunks, ignore_index=True)
            
            numeric_cols = df_file.select_dtypes(include=['float64', 'int64']).columns
            for col in numeric_cols:
                if col not in ['Date', 'Short SKU']:
                    df_file[col] = df_file[col].astype('float32')
            
            temp_file = os.path.join(temp_dir, f"processed_{i:04d}.parquet")
            df_file.to_parquet(temp_file, index=False)
            processed_files.append(temp_file)
            
            del df_file, chunks
            gc.collect()
            
    except Exception as e:
        print(f"  Error processing {file}: {e}")

print(f"\nFinished processing {len(processed_files)} files.")

# Stage 2: Combine all temporary results
print("Combining all temporary data...")

df_list = []
for f in processed_files:
    df_temp = pd.read_parquet(f)
    df_list.append(df_temp)
    del df_temp

df_combined = pd.concat(df_list, ignore_index=True)
del df_list
gc.collect()

shutil.rmtree(temp_dir)
print("Temporary files deleted.")

# Stage 3: Final transformation (melt + pivot)
print("Performing melt and pivot...")

df_long = df_combined.melt(
    id_vars=['Date', 'Short SKU'],
    var_name='Store_Code',
    value_name='Sales'
)

del df_combined
gc.collect()

df_long = df_long.dropna(subset=['Sales'])
df_long['Sales'] = pd.to_numeric(df_long['Sales'], errors='coerce').fillna(0).astype('float32')

df_pivot = df_long.pivot_table(
    index=['Store_Code', 'Short SKU'],
    columns='Date',
    values='Sales',
    aggfunc='sum',
    fill_value=0
).reset_index()

del df_long
gc.collect()

df_pivot.columns.name = None
df_pivot['Store_Code'] = df_pivot['Store_Code'].astype('category')
df_pivot['Short SKU'] = df_pivot['Short SKU'].astype(str)

print("Process completed!")
print(f"Final result: {df_pivot.shape[0]} rows, {df_pivot.shape[1]} columns")
print(f"Memory usage: {df_pivot.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Merge with SKU descriptions
df_merge = pd.merge(df_pivot, skudesc_combined, on='Short SKU', how='left')

del df_pivot, skudesc_combined
gc.collect()

print(f"Merged dataset: {df_merge.shape}")

In [ ]:
import pandas as pd
import numpy as np
import gc

def detect_date_columns(df):
    date_cols = []
    for col in df.columns:
        try:
            pd.to_datetime(col)
            date_cols.append(col)
        except:
            pass
    return date_cols

df_activedate = df_merge.copy()

sales_cols = detect_date_columns(df_activedate)
print("Detected date columns:", len(sales_cols))

# Vectorized operations instead of apply(axis=1)
df_activedate['Effective_Date'] = df_activedate['Activated Date'].fillna(df_activedate['Creation Date'])

period_arr = df_activedate['Period'].values
activated_arr = df_activedate['Activated Date'].values
effective_arr = df_activedate['Effective_Date'].values

for col in sales_cols:
    if col not in df_activedate.columns:
        continue
    
    sales_col = df_activedate[col].values
    
    cond1 = pd.isna(sales_col) & (period_arr == activated_arr)
    cond2 = pd.isna(sales_col) & (period_arr > effective_arr)
    
    df_activedate[col] = np.where(
        cond1,
        np.nan,
        np.where(cond2, 0, sales_col)
    )

df_activedate.drop(columns=['Effective_Date'], inplace=True)

del df_merge
gc.collect()

print("Active date logic completed (optimized!)")

In [ ]:
from scipy.stats import trim_mean
import numpy as np
import gc

print("Converting sales columns to numeric...")
df_activedate[sales_cols] = df_activedate[sales_cols].apply(pd.to_numeric, errors='coerce')

print("Calculating trimmed mean (ADS)...")

sales_array = df_activedate[sales_cols].values.astype('float32')

def calc_trimmed_mean_vectorized(data, cut=0.2):
    n_rows = data.shape[0]
    ads_result = np.full(n_rows, np.nan, dtype='float32')
    
    for i in range(n_rows):
        row = data[i]
        clean = row[~np.isnan(row)]
        
        if len(clean) > 0:
            sorted_clean = np.sort(clean)
            n_trim = int(len(sorted_clean) * cut)
            
            if n_trim * 2 < len(sorted_clean):
                trimmed = sorted_clean[n_trim:-n_trim] if n_trim > 0 else sorted_clean
                ads_result[i] = np.mean(trimmed)
            else:
                ads_result[i] = np.mean(clean)
    
    return ads_result

df_activedate['ADS'] = calc_trimmed_mean_vectorized(sales_array, cut=0.2)

del sales_array
gc.collect()

print("Column ADS successfully added.")

In [ ]:
import pandas as pd
import numpy as np
import gc

df_final = df_activedate.copy()

df_final['Store_Code'] = df_final['Store_Code'].astype(str)
df_final['Short SKU'] = df_final['Short SKU'].astype(str)
df_final['CODE'] = df_final['Store_Code'] + df_final['Short SKU']

df_final['ROUND'] = df_final['ADS'].round().astype('Int64')

df_final = df_final.rename(columns={
    'Store_Code': 'STORE',
    'Short SKU': 'SKU'
})

filtered_ads = df_final[~df_final['ADS'].isin([0])].copy()

required_columns = ["CODE", "STORE", "SKU", "ROUND", "ADS"]
filtered_ads = filtered_ads[required_columns]

del df_final, df_activedate
gc.collect()

output_file = "ADS OLD JUN 26 TRIM 90.csv"
filtered_ads.to_csv(output_file, index=False)

print(f"Export successful! File saved as: {output_file}")
print(f"Shape: {filtered_ads.shape}")
print(filtered_ads.head())